# Module 1 — 向量基礎與運動學

> **起點**：高中數學程度 → 物理運動學的向量描述

## 為什麼要學向量？

在日常生活中，我們常常需要同時描述「大小」和「方向」。例如：

- 你告訴朋友「走 500 公尺」，他不知道該往哪走。但如果說「往北走 500 公尺」，他就能到達目的地。
- 天氣預報說「風速 30 km/h」只告訴你風有多強，但沒說往哪吹。說「北風 30 km/h」才完整。
- GPS 導航說「距離 3 公里」沒有用，要說「東北方 3 公里」才能帶你到目的地。

**向量**就是用來描述「既有大小又有方向」的數學工具。它是整個線性代數的基礎磚塊——後面的矩陣、方程組、特徵值……全部建立在向量之上。

### 本模組涵蓋
1. 向量的定義與表示
2. 向量加法與純量乘法
3. 內積（Dot Product）與功
4. 外積（Cross Product）與力矩
5. 向量範數與距離
6. 拋體運動綜合模擬

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.linalg_utils import (
    vector_add, scalar_multiply, dot_product, cross_product,
    vector_norm, unit_vector
)
from src.physics_models import projectile_trajectory
from src.visualizer import plot_vectors_2d, plot_trajectory_2d, set_style

set_style()
print('Module 1 載入完成！')

---
## 1.1 向量的定義與表示

### 純量 vs 向量

物理量可以分成兩大類：

| 類型 | 定義 | 舉例 |
|------|------|------|
| **純量（Scalar）** | 只有大小 | 溫度 30°C、質量 5 kg、時間 3 秒、能量 100 J |
| **向量（Vector）** | 有大小 + 方向 | 速度「時速 100 km 向北」、力「50 N 向右」、位移「東方 3 km」 |

### 向量的數學表示

在平面（2D）上，一個向量可以用兩個數字表示——分別代表 x 方向和 y 方向的分量：

$$\vec{v} = \begin{bmatrix} v_x \\ v_y \end{bmatrix} = v_x \hat{i} + v_y \hat{j}$$

在空間（3D）中則需要三個數字。在 Python 中，我們用 NumPy 的 `ndarray` 來表示向量。

### 物理中的三大運動學向量

描述物體運動時，我們需要三個基本向量，它們之間透過微分/積分關聯：

$$\vec{r}(t) \xrightarrow{\text{微分}} \vec{v}(t) = \frac{d\vec{r}}{dt} \xrightarrow{\text{微分}} \vec{a}(t) = \frac{d\vec{v}}{dt}$$

- **位置向量** $\vec{r}$：物體在哪裡？（例如：「操場東方 50m、北方 30m」）
- **速度向量** $\vec{v}$：物體往哪個方向、以多快的速度移動？
- **加速度向量** $\vec{a}$：速度的變化率是多少？（例如重力加速度 $\vec{g} = [0, -9.81]$ m/s²）

> **生活例子**：開車時，儀表板上的時速就是速度的「大小」（純量），但要完整描述你的運動狀態，還需要知道你往哪個方向開（向量）。

In [ ]:
# 建立 2D 與 3D 向量
v2d = np.array([3, 4])           # 2D 向量：x 方向 3, y 方向 4
v3d = np.array([1, 2, 3])        # 3D 向量：x=1, y=2, z=3

print(f'2D 向量: {v2d}, 維度: {v2d.shape}')
print(f'3D 向量: {v3d}, 維度: {v3d.shape}')

# 物理意義：位置、速度、加速度
r = np.array([100, 200])  # 位置：東方 100m, 北方 200m
v = np.array([5, -3])     # 速度：向東 5 m/s, 向南 3 m/s
a = np.array([0, -9.81])  # 加速度：重力向下 9.81 m/s²

print(f'\n位置向量 r = {r} m       → 物體在操場東方 100m、北方 200m')
print(f'速度向量 v = {v} m/s     → 向東走 5m/s、同時向南走 3m/s')
print(f'加速度向量 a = {a} m/s² → 只有重力（向下 9.81）')

In [ ]:
# 視覺化：向量就是帶箭頭的線段
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_vectors_2d(
    [np.array([3, 4]), np.array([5, -2]), np.array([0, -9.81])],
    labels=['位置 r', '速度 v', '加速度 a'],
    colors=['blue', 'red', 'green'],
    ax=ax1,
    title='位置、速度、加速度向量'
)

# 向量的分量表示：任何向量都可以拆成 x 和 y 分量的「合體」
plot_vectors_2d(
    [np.array([4, 0]), np.array([0, 3]), np.array([4, 3])],
    labels=['x 分量=4', 'y 分量=3', '合向量(4,3)'],
    colors=['orange', 'purple', 'black'],
    ax=ax2,
    title='向量 = x 分量 + y 分量'
)
plt.tight_layout()
plt.show()

---
## 1.2 向量加法與純量乘法

### 原理

**向量加法**就是把兩個向量「接力」：第一個向量走完之後，從終點出發走第二個向量。

$$\vec{a} + \vec{b} = \begin{bmatrix} a_x + b_x \\ a_y + b_y \end{bmatrix}$$

幾何上有兩種理解方式：
- **三角形法則**：兩向量頭尾相接，從起點到終點的向量就是和
- **平行四邊形法則**：兩向量從同一點出發，對角線就是和

**純量乘法**就是「放大或縮小」向量：
- `2 * v` → 方向不變、長度加倍
- `-1 * v` → 方向反轉、長度不變
- `0.5 * v` → 方向不變、長度減半

### 實際例子：飛機速度合成

飛機在空中飛行時，實際的「地面速度」等於飛機本身的速度加上風速：

$$\vec{v}_{\text{地面}} = \vec{v}_{\text{飛機}} + \vec{v}_{\text{風}}$$

這就是為什麼航班在順風時比較快，逆風時比較慢。航空公司每天都在用向量加法來計算飛行時間和油耗。

### 實際例子：斜面力分解

一個放在斜面上的箱子，受到向下的重力 $mg$。但箱子不是直接掉下去，而是沿著斜面滑。
這是因為重力可以「分解」成兩個分量：
- **法向分力**（垂直斜面）= $mg\cos\theta$ → 被斜面支撐住了
- **切線分力**（沿著斜面）= $mg\sin\theta$ → 讓箱子滑動的力

這就是向量加法的逆運算——把一個向量拆成兩個互相垂直的分量。

In [ ]:
# === 飛機速度合成 ===
# 想像一架從台北飛往東京的飛機
v_plane = np.array([200, 0])     # 飛機向東飛 200 km/h
v_wind = np.array([-30, 50])     # 西南風：逆風 30 + 側風 50 km/h
v_ground = vector_add(v_plane, v_wind)  # 地面觀測到的實際速度

print('=== 飛機速度合成 ===')
print(f'飛機空速: {v_plane} km/h（向東 200）')
print(f'風速:     {v_wind} km/h（逆風 30 + 北風 50）')
print(f'地面速度: {v_ground} km/h')
print(f'地面速率: {vector_norm(v_ground):.1f} km/h（比空速 200 慢！因為有逆風）')

# 視覺化——三角形法則
fig, ax = plt.subplots(figsize=(10, 8))
ax.annotate('', xy=v_plane, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.annotate('', xy=v_ground, xytext=v_plane,
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.annotate('', xy=v_ground, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax.text(100, -15, '飛機空速', color='blue', fontsize=12, fontweight='bold')
ax.text(175, 30, '風速', color='red', fontsize=12, fontweight='bold')
ax.text(70, 35, '地面速度（實際）', color='green', fontsize=12, fontweight='bold')
ax.set_xlim(-50, 250)
ax.set_ylim(-50, 80)
ax.set_aspect('equal')
ax.set_xlabel('東方 (km/h)')
ax.set_ylabel('北方 (km/h)')
ax.set_title('飛機速度合成 — 向量加法的三角形法則')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === 斜面力分解 ===
# 一個 10 kg 的箱子放在 30° 的斜面上
theta = np.radians(30)  # 斜面角度
m = 10                  # 質量 10 kg
g = 9.81                # 重力加速度
mg = m * g              # 重力 = 98.1 N

F_normal = mg * np.cos(theta)   # 法向分力（被斜面擋住）
F_tangent = mg * np.sin(theta)  # 切線分力（讓箱子滑動）

print('=== 斜面力分解 ===')
print(f'箱子質量: {m} kg，重力: {mg:.1f} N')
print(f'斜面角度: {np.degrees(theta):.0f}°')
print(f'法向分力: {F_normal:.1f} N（垂直斜面，被支撐住）')
print(f'切線分力: {F_tangent:.1f} N（沿斜面向下，讓箱子滑動）')
print(f'\n驗證：分力的平方和 = 重力的平方')
print(f'  √({F_normal:.1f}² + {F_tangent:.1f}²) = {np.sqrt(F_normal**2 + F_tangent**2):.1f} N = {mg:.1f} N ✓')
print(f'\n生活應用：如果斜面有摩擦力 μ = 0.3')
friction = 0.3 * F_normal
net_force = F_tangent - friction
print(f'  摩擦力 = μ × N = 0.3 × {F_normal:.1f} = {friction:.1f} N')
print(f'  淨力 = {F_tangent:.1f} - {friction:.1f} = {net_force:.1f} N → 箱子{"會滑動" if net_force > 0 else "靜止不動"}')

# 視覺化斜面力分解
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot([0, 5*np.cos(theta)], [0, 5*np.sin(theta)], 'k-', lw=3)
ax.fill_between([0, 5*np.cos(theta)], [0, 0], [0, 5*np.sin(theta)], alpha=0.1, color='gray')
obj_x, obj_y = 2.5*np.cos(theta), 2.5*np.sin(theta)
ax.plot(obj_x, obj_y, 'ks', ms=15)
scale = 0.03
ax.annotate('', xy=(obj_x, obj_y - mg*scale), xytext=(obj_x, obj_y),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(obj_x + 0.1, obj_y - mg*scale/2, f'mg={mg:.0f}N', color='blue', fontsize=11)
nx, ny = -np.sin(theta), np.cos(theta)
ax.annotate('', xy=(obj_x - F_normal*scale*nx, obj_y - F_normal*scale*ny),
            xytext=(obj_x, obj_y), arrowprops=dict(arrowstyle='->', color='red', lw=2))
tx, ty = np.cos(theta), np.sin(theta)
ax.annotate('', xy=(obj_x - F_tangent*scale*tx, obj_y - F_tangent*scale*ty),
            xytext=(obj_x, obj_y), arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(obj_x - 1.8, obj_y + 0.3, f'法向力 N={F_normal:.0f}N', color='red', fontsize=11)
ax.text(obj_x - 1.2, obj_y - 1.2, f'切線力 F={F_tangent:.0f}N', color='green', fontsize=11)
ax.set_xlim(-1, 6)
ax.set_ylim(-1.5, 4)
ax.set_aspect('equal')
ax.set_title(f'斜面力分解（θ = {np.degrees(theta):.0f}°，{m} kg 箱子）')
ax.grid(True, alpha=0.3)
plt.show()

---
## 1.3 內積（Dot Product）

### 原理

內積是兩個向量之間最基本的「互動方式」，它把兩個向量變成一個純量（數字）。

**代數定義**（逐元素相乘再求和）：
$$\vec{a} \cdot \vec{b} = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n = \sum_{i=1}^n a_i b_i$$

**幾何定義**（用夾角表示）：
$$\vec{a} \cdot \vec{b} = |\vec{a}| \cdot |\vec{b}| \cdot \cos\theta$$

其中 $\theta$ 是兩向量之間的夾角。

### 直覺理解

內積可以理解為「一個向量在另一個向量方向上的投影長度，再乘以另一個向量的長度」。

- 兩向量同方向（$\theta = 0°$）：$\cos 0° = 1$ → 內積最大（完全對齊）
- 兩向量垂直（$\theta = 90°$）：$\cos 90° = 0$ → 內積為零（完全無關）
- 兩向量反方向（$\theta = 180°$）：$\cos 180° = -1$ → 內積為負（對抗）

### 物理應用：功 $W = \vec{F} \cdot \vec{d}$

在物理中，「功」（Work）是力對物體做的貢獻。但不是所有的力都做功——只有**沿著位移方向**的分力才做功。

$$W = \vec{F} \cdot \vec{d} = |F| \cdot |d| \cdot \cos\theta$$

- 你推箱子向前，力和位移同向 → 做正功（消耗能量）
- 你提著箱子走路，力向上但位移向前（垂直）→ 做功為零！（這就是為什麼提重物走路不算「做功」）
- 摩擦力和運動方向相反 → 做負功（消耗動能）

In [ ]:
# === 功的計算：推箱子 ===
# 你用 50N 的力，以 30° 向下的角度推一個箱子前進 10m
F = np.array([50, 30])     # 力向量 (N)：水平50N + 垂直30N
d = np.array([10, 0])      # 位移向量 (m)：水平移動10m

W = dot_product(F, d)
W_numpy = np.dot(F, d)

print('=== 推箱子做功 ===')
print(f'力 F = {F} N（水平 50N + 向上 30N）')
print(f'位移 d = {d} m（水平 10m）')
print(f'功 W = F·d = {W} J')
print(f'\n為什麼是 500J 而不是 583J？')
print(f'因為只有水平分力做功：50N × 10m = 500J')
print(f'垂直分力 30N 的方向和位移垂直，所以不做功！')
print(f'\n手動實作 vs NumPy: {W} == {W_numpy} ✓')

# 夾角計算
cos_theta = dot_product(F, d) / (vector_norm(F) * vector_norm(d))
theta = np.arccos(cos_theta)
print(f'\n力與位移的夾角: {np.degrees(theta):.1f}°')
print(f'W = |F|·|d|·cosθ = {vector_norm(F):.1f} × {vector_norm(d):.1f} × cos({np.degrees(theta):.1f}°) = {W:.0f} J ✓')

In [ ]:
# === 磁力永不做功 ===
# 帶電粒子在磁場中運動時，磁力永遠垂直於速度方向
# 就像旋轉木馬的繩子拉力——永遠不會讓你跑得更快或更慢，只是改變方向

t = np.linspace(0, 2*np.pi, 100)
r = 1.0
x, y = r * np.cos(t), r * np.sin(t)
vx, vy = -r * np.sin(t), r * np.cos(t)  # 速度（圓的切線方向）
Fx, Fy = -np.cos(t), -np.sin(t)          # 磁力（指向圓心）

power = Fx * vx + Fy * vy  # 瞬時功率 = F·v
print('=== 磁力永不做功 ===')
print(f'磁力做功率: max|P| = {np.max(np.abs(power)):.2e} ≈ 0')
print(f'物理解釋：磁力始終垂直於速度方向 → F·v = 0')
print(f'就像用繩子轉石頭：繩子的張力只改變方向，不改變速率')
print(f'這就是為什麼磁鐵靠近鐵釘時，鐵釘的動能不會改變')

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(x, y, 'b-', lw=2)
step = 10
scale = 0.3
for i in range(0, len(t), step):
    ax.annotate('', xy=(x[i]+vx[i]*scale, y[i]+vy[i]*scale),
                xytext=(x[i], y[i]), arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    ax.annotate('', xy=(x[i]+Fx[i]*scale, y[i]+Fy[i]*scale),
                xytext=(x[i], y[i]), arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.plot([], [], 'r-', lw=2, label='速度 v（切線方向）')
ax.plot([], [], 'g-', lw=2, label='磁力 F（向心方向）')
ax.legend(fontsize=12)
ax.set_aspect('equal')
ax.set_title('磁力永不做功：F 永遠垂直於 v → F·v = 0')
ax.grid(True, alpha=0.3)
plt.show()

### 實際例子：太陽能板最佳安裝角度

太陽能板的發電量與「光線照射角度」有關。當光線垂直打在面板上時效率最高。

數學上，發電效率正比於**太陽光方向**與**面板法向量**的內積：

$$\text{效率} \propto \vec{n}_{\text{panel}} \cdot \vec{d}_{\text{sun}} = \cos\theta$$

當 $\theta = 0$（面板法向量正對太陽）時效率最大。

> **台灣實例**：台灣位於北緯 23-25°，夏天太陽仰角約 88°（接近垂直），冬天約 43°。因此太陽能板通常安裝在朝南約 23° 傾斜的角度，以最大化全年日照。

In [ ]:
# === 太陽能板最佳角度 ===
sun_elevation = np.radians(60)  # 正午太陽仰角 60°
sun_direction = np.array([np.cos(sun_elevation), np.sin(sun_elevation)])

panel_angles = np.linspace(0, 90, 91)
efficiencies = []
for angle in panel_angles:
    panel_normal = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
    efficiency = dot_product(sun_direction, panel_normal)
    efficiencies.append(max(0, efficiency))

best_angle = panel_angles[np.argmax(efficiencies)]
print(f'太陽仰角: {np.degrees(sun_elevation):.0f}°')
print(f'太陽能板最佳傾斜角: {best_angle:.0f}°')
print(f'原因：面板法向量要對準太陽方向 → 最佳傾斜角 = 太陽仰角')
print(f'\n延伸：在台北（北緯 25°），冬至太陽仰角約 41.5°')
print(f'如果只能固定一個角度，通常選 25° 左右（全年平均最佳）')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(panel_angles, efficiencies, 'b-', lw=2)
ax.axvline(best_angle, color='r', ls='--', label=f'最佳角度 = {best_angle:.0f}°')
ax.fill_between(panel_angles, efficiencies, alpha=0.1, color='blue')
ax.set_xlabel('太陽能板傾斜角 (°)')
ax.set_ylabel('效率 (cos θ)')
ax.set_title('太陽能板傾斜角 vs 發電效率（內積的應用）')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

---
## 1.4 外積（Cross Product）

### 原理

外積是 **3D 向量專屬**的運算。它把兩個向量變成一個**新的向量**（不是純量）。

$$\vec{a} \times \vec{b} = \begin{vmatrix} \hat{i} & \hat{j} & \hat{k} \\ a_1 & a_2 & a_3 \\ b_1 & b_2 & b_3 \end{vmatrix} = \begin{bmatrix} a_2 b_3 - a_3 b_2 \\ a_3 b_1 - a_1 b_3 \\ a_1 b_2 - a_2 b_1 \end{bmatrix}$$

### 外積的三個重要性質

1. **結果向量垂直於原來兩個向量**（用右手定則判斷方向）
2. **大小 = 兩向量構成的平行四邊形面積**：$|\vec{a} \times \vec{b}| = |a||b|\sin\theta$
3. **反交換律**：$\vec{a} \times \vec{b} = -(\vec{b} \times \vec{a})$（順序很重要！）

### 物理應用：力矩 $\vec{\tau} = \vec{r} \times \vec{F}$

力矩描述的是「旋轉的能力」。用扳手鎖螺絲為例：
- $\vec{r}$ = 從旋轉軸到施力點的向量（扳手的長度方向）
- $\vec{F}$ = 你施加的力
- $\vec{\tau}$ = 力矩 = 讓螺絲轉動的效果

$$|\tau| = |r| \cdot |F| \cdot \sin\theta$$

- 垂直施力（$\theta = 90°$）→ $\sin 90° = 1$ → 力矩最大（最有效率）
- 沿著扳手推（$\theta = 0°$）→ $\sin 0° = 0$ → 力矩為零（白費力氣）

> **生活經驗**：開門時，你推門把（離門軸最遠）比推門中間更省力——因為 $|r|$ 更大。同理，長扳手比短扳手更容易鎖螺絲。

In [ ]:
# === 扳手力矩 ===
wrench_length = 0.3  # 30 cm 扳手
F_magnitude = 50      # 50 N

angles = np.linspace(0, 180, 181)
torques = []
for angle in angles:
    rad = np.radians(angle)
    r = np.array([wrench_length, 0, 0])
    F = np.array([F_magnitude * np.cos(rad), F_magnitude * np.sin(rad), 0])
    tau = cross_product(r, F)
    torques.append(vector_norm(tau))

print('=== 扳手鎖螺絲 ===')
print(f'扳手長度: {wrench_length*100:.0f} cm')
print(f'施力: {F_magnitude} N')
print(f'\n90° 垂直施力 → 力矩: {torques[90]:.1f} N·m（最大，最有效率！）')
print(f'45° 斜向施力 → 力矩: {torques[45]:.1f} N·m（只有 70% 效率）')
print(f' 0° 沿扳手推 → 力矩: {torques[0]:.4f} N·m ≈ 0（完全白費力氣）')
print(f'\n驗證: 90° 理論值 = r × F = {wrench_length} × {F_magnitude} = {wrench_length*F_magnitude:.1f} N·m ✓')
print(f'\n實用技巧：如果螺絲太緊，換一支更長的扳手（增加 r）')
print(f'  60cm 扳手的力矩 = {0.6*F_magnitude:.0f} N·m → 是 30cm 扳手的 2 倍！')

# 驗證與 NumPy 一致
r_test = np.array([0.3, 0, 0])
F_test = np.array([0, 50, 0])
assert np.allclose(cross_product(r_test, F_test), np.cross(r_test, F_test))
print(f'\n手動實作與 np.cross 一致 ✓')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(angles, torques, 'b-', lw=2)
ax.axvline(90, color='r', ls='--', label='90°（最大力矩）')
ax.fill_between(angles, torques, alpha=0.1, color='blue')
ax.set_xlabel('施力角度 (°)')
ax.set_ylabel('力矩 |τ| (N·m)')
ax.set_title('扳手效率 — 垂直施力最有效')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === 驗證外積大小 = 平行四邊形面積 ===
a = np.array([3, 0, 0])  # 長度 3 的水平向量
b_angle = np.pi / 6       # 與 a 夾 30°
b = np.array([2 * np.cos(b_angle), 2 * np.sin(b_angle), 0])  # 長度 2

cross = cross_product(a, b)
area_cross = vector_norm(cross)
area_formula = vector_norm(a) * vector_norm(b) * np.sin(b_angle)

print('=== 外積大小 = 平行四邊形面積 ===')
print(f'向量 a: 長度 {vector_norm(a):.0f}')
print(f'向量 b: 長度 {vector_norm(b):.0f}，與 a 夾角 30°')
print(f'|a × b| = {area_cross:.4f}')
print(f'|a|·|b|·sin30° = {area_formula:.4f}')
print(f'兩者相等 ✓')
print(f'\n幾何意義：以 a 和 b 為邊的平行四邊形面積 = {area_cross:.1f} 平方單位')
print(f'（三角形面積就是平行四邊形的一半 = {area_cross/2:.1f}）')

---
## 1.5 向量範數與距離

### 原理

**範數**（Norm）就是衡量向量「有多長」的方式。但「長度」可以有不同的定義方式：

| 範數 | 數學公式 | 直覺理解 | 生活例子 |
|------|----------|----------|----------|
| **L1（曼哈頓）** | $\sum\|v_i\|$ | 只走水平/垂直的總距離 | 在棋盤格街道走路（如曼哈頓市區） |
| **L2（歐幾里得）** | $\sqrt{\sum v_i^2}$ | 兩點間直線距離 | 鳥飛的直線距離 |
| **L∞（無窮）** | $\max\|v_i\|$ | 最大的分量 | 西洋棋國王走一步的距離 |

### 單位向量

**單位向量**是長度為 1 的向量，只保留「方向」資訊：

$$\hat{v} = \frac{\vec{v}}{|\vec{v}|}$$

就像指南針的箭頭——它不告訴你距離多遠，只告訴你方向。

> **工程應用**：機器手臂的控制器需要知道目標在「哪個方向」（單位向量）和「多遠」（範數），才能規劃路徑。

In [ ]:
# === 三種範數比較 ===
v = np.array([3, 4])
l1 = vector_norm(v, 1)
l2 = vector_norm(v, 2)
linf = vector_norm(v, np.inf)

print('=== 向量 v = [3, 4] 的三種範數 ===')
print(f'L1 範數: {l1:.0f}（從原點走到(3,4)，只能走直角路線 = 3+4 = 7 步）')
print(f'L2 範數: {l2:.0f}（直線距離 = √(9+16) = 5，就是畢氏定理！）')
print(f'L∞ 範數: {linf:.0f}（最大分量 = max(3,4) = 4）')
print(f'\n關係: L∞ ≤ L2 ≤ L1（永遠成立）')
print(f'       {linf:.0f} ≤ {l2:.0f} ≤ {l1:.0f} ✓')

assert l1 == np.linalg.norm(v, 1)
assert l2 == np.linalg.norm(v, 2)
assert linf == np.linalg.norm(v, np.inf)
print(f'\n手動實作與 np.linalg.norm 一致 ✓')

In [ ]:
# === GPS 軌跡總里程 ===
# 模擬跑步 GPS 紀錄：實際跑的路線比直線距離長
np.random.seed(42)
n_points = 50
t = np.linspace(0, 2*np.pi, n_points)
gps_x = 100 * t + 10 * np.sin(3*t) + np.random.normal(0, 2, n_points)
gps_y = 50 * np.sin(t) + np.random.normal(0, 2, n_points)

# 計算總里程 = 相鄰 GPS 點的距離加總
total_distance = 0
for i in range(1, n_points):
    segment = np.array([gps_x[i] - gps_x[i-1], gps_y[i] - gps_y[i-1]])
    total_distance += vector_norm(segment)  # L2 範數 = 直線距離

straight = vector_norm(np.array([gps_x[-1]-gps_x[0], gps_y[-1]-gps_y[0]]))

print('=== GPS 軌跡分析 ===')
print(f'GPS 記錄點數: {n_points}')
print(f'實際跑步里程: {total_distance:.1f} m（把每小段 L2 距離加起來）')
print(f'起點到終點直線: {straight:.1f} m')
print(f'迂迴率: {total_distance/straight:.2f}（跑了直線距離的 {total_distance/straight:.1f} 倍）')
print(f'\n應用：Garmin、Apple Watch 就是這樣計算你的跑步距離的！')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gps_x, gps_y, 'b.-', lw=1, ms=4)
ax.plot([gps_x[0], gps_x[-1]], [gps_y[0], gps_y[-1]], 'r--', lw=2, label=f'直線 {straight:.0f}m')
ax.plot(gps_x[0], gps_y[0], 'go', ms=10, label='起點')
ax.plot(gps_x[-1], gps_y[-1], 'rs', ms=10, label='終點')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title(f'GPS 軌跡 — 跑步里程 {total_distance:.0f}m vs 直線 {straight:.0f}m')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === 單位向量：只保留方向 ===
# 飛行器的飛行方向
direction = np.array([3, 4, 1])  # 飛行方向（含高度分量）
u = unit_vector(direction)

print('=== 單位向量 ===')
print(f'飛行方向: {direction}')
print(f'單位向量: [{u[0]:.4f}, {u[1]:.4f}, {u[2]:.4f}]')
print(f'長度 = {vector_norm(u):.10f}（精確等於 1.0）✓')
print(f'\n用途：飛行控制器用單位向量表示「方向」，再乘以速率表示「速度」')
print(f'  如果飛行速率是 200 m/s:')
v_flight = 200 * u
print(f'  速度向量 = 200 × û = [{v_flight[0]:.1f}, {v_flight[1]:.1f}, {v_flight[2]:.1f}] m/s')

---
## 1.6 拋體運動完整模擬

### 原理

拋體運動是向量概念的完美綜合應用。一個被拋出的物體（忽略空氣阻力）的運動可以分成兩個獨立的分量：

- **水平方向**：沒有力作用 → 等速運動（$v_x$ 恆定）
- **垂直方向**：只有重力 → 等加速運動（$v_y$ 隨時間遞減）

$$\vec{r}(t) = \begin{bmatrix} v_0 \cos\theta \cdot t \\ v_0 \sin\theta \cdot t - \frac{1}{2}g t^2 \end{bmatrix}$$

### 為什麼 45° 射程最大？

射程公式為 $R = \frac{v_0^2 \sin(2\theta)}{g}$，當 $2\theta = 90°$，即 $\theta = 45°$ 時 $\sin$ 值最大。

直覺理解：角度太小 → 雖然水平速度大但飛行時間短；角度太大 → 雖然飛得高但水平速度小。45° 是兩者的最佳折衷。

> **實際例子**：棒球外野手傳球、足球員射門、砲兵計算彈道、奧運標槍選手（實際最佳角度約 34° 因為空氣阻力和出手高度）。

In [ ]:
# === 拋體運動模擬 ===
v0 = 30  # 初速 30 m/s
angles_deg = [15, 30, 45, 60, 75]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for angle_deg in angles_deg:
    theta = np.radians(angle_deg)
    t, x, y, vx, vy = projectile_trajectory(v0, theta)
    ax1.plot(x, y, lw=2, label=f'{angle_deg}°（射程 {x[-1]:.0f}m）')

ax1.set_xlabel('水平距離 (m)')
ax1.set_ylabel('高度 (m)')
ax1.set_title(f'拋體軌跡比較（v₀ = {v0} m/s）')
ax1.legend(fontsize=10)
ax1.set_ylim(bottom=0)
ax1.grid(True, alpha=0.3)

# 45° 的速度向量
t, x, y, vx, vy = projectile_trajectory(v0, np.radians(45))
plot_trajectory_2d(x, y, vx, vy, ax=ax2, title='45° 拋射 — 速度向量隨時間變化', arrow_step=20)
plt.tight_layout()
plt.show()

# 驗證 45° 最大
ranges = {}
for angle_deg in range(1, 90):
    _, x_traj, _, _, _ = projectile_trajectory(v0, np.radians(angle_deg))
    ranges[angle_deg] = x_traj[-1]

best = max(ranges, key=ranges.get)
print(f'\n數值驗證：最大射程角度 = {best}°，射程 = {ranges[best]:.1f} m')
print(f'理論公式：R = v₀²/g = {v0}²/9.81 = {v0**2/9.81:.1f} m')
print(f'\n觀察：15° 和 75° 的射程幾乎相同（互補角射程相等）')
print(f'  15°: {ranges[15]:.1f}m  vs  75°: {ranges[75]:.1f}m')
print(f'  30°: {ranges[30]:.1f}m  vs  60°: {ranges[60]:.1f}m')

---
## Module 1 驗證總結

| 項目 | 驗證方式 | 物理意義 | 結果 |
|------|----------|----------|------|
| 向量加法 | `np.allclose` 比對 | 飛機地面速度 = 空速 + 風速 | ✓ |
| 斜面力分解 | 分力平方和 = 重力平方 | 法向力 + 切線力 = mg | ✓ |
| 內積 | `np.dot` 比對 | 功 W = F·d | ✓ |
| 磁力做功 | F·v = 0 | 磁力永遠垂直速度 | ✓ |
| 太陽能板 | 最佳角 = 太陽仰角 | cos θ 最大化 | ✓ |
| 外積 | `np.cross` 比對 | 力矩 τ = r × F | ✓ |
| 扳手力矩 | 90° 最大 | sinθ 最大化 | ✓ |
| 範數 | `np.linalg.norm` 比對 | GPS 里程計算 | ✓ |
| 拋體運動 | 45° 射程最大 | 解析解 R = v²/g | ✓ |

### 本模組重點回顧

- **向量**用來描述有大小又有方向的物理量
- **向量加法**就是分量相加，物理上是力的合成、速度的合成
- **內積**衡量兩向量的「對齊程度」，物理上是功的計算
- **外積**產生垂直於兩向量的新向量，物理上是力矩的計算
- **範數**衡量向量的長度，是計算距離的基礎